# Module 1 Milestone: Data facts, an LLM summary, saved to JSON


In [1]:
import pandas as pd
from io import StringIO

csv_text = """product,category,units,revenue
Widget,hardware,120,2400
Gadget,hardware,80,3200
Cable,accessory,300,1500
Case,accessory,150,1200
App,software,50,5000"""

df = pd.read_csv(StringIO(csv_text))

print(df)

  product   category  units  revenue
0  Widget   hardware    120     2400
1  Gadget   hardware     80     3200
2   Cable  accessory    300     1500
3    Case  accessory    150     1200
4     App   software     50     5000


## 1. Compute facts

In [2]:
def compute_facts(df: pd.DataFrame) -> dict:
    rows = len(df)
    total_revenue = int(df["revenue"].sum())

    category_totals = df.groupby("category")["revenue"].sum()
    top_category = category_totals.idxmax()

    average_units = float(df["units"].mean())

    facts = {
        "rows": rows,
        "total_revenue": total_revenue,
        "top_category_by_revenue": top_category,
        "average_units": average_units
    }

    return facts


facts = compute_facts(df)
print(facts)

{'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'hardware', 'average_units': 140.0}


## 2. Build a prompt from the facts

In [3]:
import json

def build_prompt(facts: dict) -> str:
    prompt = (
        "Summarize these dataset facts in a short and clear paragraph:\n"
        f"- Rows: {facts['rows']}\n"
        f"- Total revenue: {facts['total_revenue']}\n"
        f"- Top category by revenue: {facts['top_category_by_revenue']}\n"
        f"- Average units: {facts['average_units']}"
    )

    return prompt


prompt = build_prompt(facts)
print(prompt)

Summarize these dataset facts in a short and clear paragraph:
- Rows: 5
- Total revenue: 13300
- Top category by revenue: hardware
- Average units: 140.0


## 3. Ask the LLM


In [5]:
import requests

def ask_llm(prompt: str) -> str:
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3.2",
                "prompt": prompt,
                "stream": False
            },
            timeout=30
        )

        response.raise_for_status()

        result = response.json()
        return result["response"]

    except Exception:
        return "LLM summary unavailable. Facts were computed successfully."


summary = ask_llm(prompt)
print(summary)

/Users/haripriyareddypatil/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


This dataset contains 5 entries, with a total revenue of $13,300 across all categories. Notably, the top category in terms of revenue is hardware, while the average number of units sold is 140.0.


## 4. Save facts and summary to JSON

In [6]:
class Report:
    def __init__(self, path: str):
        self.path = path

    def save(self, facts: dict, summary: str) -> None:
        report_data = {
            "facts": facts,
            "summary": summary
        }

        with open(self.path, "w") as file:
            json.dump(report_data, file, indent=4)


report = Report("report.json")
report.save(facts, summary)

print("Saved report.json")

Saved report.json


## 5. Check the saved JSON

In [7]:
with open("report.json", "r") as file:
    saved_report = json.load(file)

print(saved_report)

{'facts': {'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'hardware', 'average_units': 140.0}, 'summary': 'This dataset contains 5 entries, with a total revenue of $13,300 across all categories. Notably, the top category in terms of revenue is hardware, while the average number of units sold is 140.0.'}
